# 🩺 Guardiã AI - Inteligência Artificial para Saúde e Segurança da Mulher

Este notebook contém o código-fonte do "Guardiã AI - Inteligência Artificial para Saúde e Segurança da Mulher", solução desenvolvida no âmbito do desafio (Tech Challenge) apresentado durante a Quinta Fase da Pós Tech (8IADT), da Faculdade de Informática e Administração Paulista (FIAP), conforme requisitos contidos no PDF disponível no seguinte repositório: https://github.com/marceloklotz/fiap-quinta-fase

O desafio propõe a criação do Guardiã AI - Inteligência Artificial para Saúde e Segurança da Mulher, uma aplicação capaz de receber informações de um atendimento e utilizar Inteligência Artificial para auxiliar na análise inicial do caso, auxiliando às equipes profissionais atuantes em questões voltadas à saúde da mulher e à segurança da mulher.

⚠️ **Aviso de Responsabilidade (Disclaimer)**: Este projeto foi desenvolvido exclusivamente para fins educacionais. Ressaltamos que a ferramenta não substitui validações médicas e não está apta para embasar decisões clínicas, realizar diagnósticos, apoiar triagens ou auxiliar vítimas em situações de emergência. Toda e qualquer interpretação dos resultados deve ficar a cargo de profissionais competentes.

# 1. Instalação de Dependências
Este bloco instala todas as bibliotecas necessárias para rodar o pipeline completo do Guardiã AI. Ele inclui pacotes para transcrição de áudio (openai-whisper), processamento de sinais (librosa), Inteligência Artificial Generativa e RAG (langchain, faiss-cpu), geração de dados sintéticos (faker) e Machine Learning/Explicabilidade (xgboost, shap, scikit-learn).

In [1]:
!pip install openai-whisper librosa deep-translator transformers langchain-core langchain-openai langchain-community tiktoken faiss-cpu faker xgboost shap scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 49.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.4/127.4 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 106.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.2 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none

# 2. Importação de Bibliotecas e Configuração de Ambiente
Aqui carregamos os módulos essenciais e configuramos a chave de API da OpenAI de forma segura utilizando a biblioteca getpass, evitando que a credencial fique exposta no código.

In [2]:
import os
import getpass
import pandas as pd
import numpy as np
import random
import re
import glob
from datetime import datetime
import xgboost as xgb
import shap
import librosa
import whisper
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, f1_score, confusion_matrix
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.callbacks import get_openai_callback
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Configuração da API Key
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Insira sua OpenAI API Key: ")

/tmp/ipykernel_694/2574290808.py:17: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.callbacks import get_openai_callback


Insira sua OpenAI API Key: ··········


# 2.5 Montagem do Google Drive
Célula dedicada a montar o sistema de arquivos do Google Drive no ambiente Colab, permitindo que a solução recupere os arquivos Markdown (.md) de protocolos PCDTs armazenados no drive do usuário.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 3. Motor de Geração de Dataset Sintético

Este script cria um dataset tabular fictício de 5.000 pacientes para treinar o modelo de Machine Learning. Utiliza a biblioteca Faker (com limpeza de prefixos via Regex) para dados demográficos e aplica distribuições normais (numpy) para simular sinais vitais clinicamente coerentes. A variável alvo urgencia_critica é gerada com base em uma regra médica determinística. O arquivo é salvo localmente no Colab.

In [4]:
from faker import Faker
fake = Faker('pt_BR')
Faker.seed(42)
np.random.seed(42)

num_registros = 5000

# Função para remover abreviaturas e pronomes de tratamento
def gerar_nome_limpo():
    nome = fake.name()
    return re.sub(r'^(Sr\.|Sra\.|Srta\.|Dr\.|Dra\.|Prof\.|Profa\.)\s+', '', nome)

dados = {
    'nome_completo': [gerar_nome_limpo() for _ in range(num_registros)],
    'data_nascimento': [fake.date_of_birth(minimum_age=18, maximum_age=90) for _ in range(num_registros)],
    'rg': [fake.rg() for _ in range(num_registros)],
    'cpf': [fake.cpf() for _ in range(num_registros)],
    'historico_comorbidades': [random.choice(['Nenhuma', 'Hipertensão', 'Diabetes', 'Doença Cardiovascular', 'Asma']) for _ in range(num_registros)]
}
df = pd.DataFrame(dados)
df['idade'] = df['data_nascimento'].apply(lambda x: (datetime.now().date() - x).days // 365)

# Geração de Sinais Vitais (Distribuições Normais)
df['pressao_sistolica'] = np.clip(np.random.normal(120, 20, num_registros), 70, 220).astype(int)
df['pressao_diastolica'] = np.clip(np.random.normal(80, 15, num_registros), 40, 130).astype(int)
df['frequencia_cardiaca'] = np.clip(np.random.normal(85, 25, num_registros), 40, 180).astype(int)
df['frequencia_respiratoria'] = np.clip(np.random.normal(18, 5, num_registros), 10, 40).astype(int)
df['temperatura_celsius'] = np.round(np.clip(np.random.normal(36.5, 0.8, num_registros), 34.0, 41.0), 1)
df['spo2_porcento'] = np.clip(np.random.normal(97, 4, num_registros), 70, 100).astype(int)
df['taxa_hesitacao_porcento'] = np.round(np.random.uniform(0, 100, num_registros), 2)
df['possui_comorbidade'] = np.where(df['historico_comorbidades'] == 'Nenhuma', 0, 1)

# Regra Alvo: Classificação de Urgência
condicao_risco = (df['spo2_porcento'] < 93) | ((df['taxa_hesitacao_porcento'] > 30) & (df['frequencia_cardiaca'] > 115))
df['urgencia_critica'] = np.where(condicao_risco, 1, 0)

# Exportação
df.to_csv('/content/dataset_guardia_ai_triagem.csv', index=False)
print("✅ Dataset sintético gerado e salvo em /content/dataset_guardia_ai_triagem.csv")

✅ Dataset sintético gerado e salvo em /content/dataset_guardia_ai_triagem.csv


# 4. Treinamento do Classificador (XGBoost) e Explicabilidade (SHAP)
Esta etapa carrega o dataset gerado, divide em treino e teste, e treina um modelo XGBClassifier balanceando os pesos das classes. Em seguida, extrai métricas críticas (Recall e F1-Score). A função analisar_risco_shap é definida para gerar predições em tempo real e usar o TreeExplainer para traduzir o peso matemático das features (SHAP values) em uma justificativa textual clara.

In [5]:
features = [
    'idade', 'pressao_sistolica', 'pressao_diastolica', 'frequencia_cardiaca',
    'frequencia_respiratoria', 'temperatura_celsius', 'spo2_porcento',
    'taxa_hesitacao_porcento', 'possui_comorbidade'
]
X = df[features]
y = df['urgencia_critica']

# Divisão estratificada para lidar com desbalanceamento
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Treinamento do XGBoost
modelo_xgb = xgb.XGBClassifier(
    objective='binary:logistic', n_estimators=100, learning_rate=0.1, max_depth=5,
    scale_pos_weight=len(y[y==0])/len(y[y==1]), random_state=42
)
modelo_xgb.fit(X_train, y_train)

y_pred = modelo_xgb.predict(X_test)
print(f"✅ Classificador XGBoost Treinado.")
print(f"Recall: {recall_score(y_test, y_pred):.4f} | F1-Score: {f1_score(y_test, y_pred):.4f}")

# Inicialização do SHAP
explainer = shap.TreeExplainer(modelo_xgb)

def analisar_risco_shap(dados_paciente_dict):
    df_paciente = pd.DataFrame([dados_paciente_dict])[features]
    prob = modelo_xgb.predict_proba(df_paciente)[0][1]
    classe = modelo_xgb.predict(df_paciente)[0]

    # Extração de pesos
    shap_vals = explainer.shap_values(df_paciente)[0]
    feature_imp = pd.DataFrame({'Feature': features, 'Valor': df_paciente.iloc[0].values, 'SHAP': shap_vals})
    agravantes = feature_imp[feature_imp['SHAP'] > 0].sort_values(by='SHAP', ascending=False).head(3)

    nivel = "Risco Alto" if classe == 1 else "Risco Baixo"
    texto_shap = f"Classificação algorítmica: {nivel} ({prob*100:.1f}%). "

    if classe == 1 and not agravantes.empty:
        motivos = [f"{row['Feature']} ({row['Valor']})" for _, row in agravantes.iterrows()]
        texto_shap += "Justificativa matemática (SHAP): O risco foi elevado criticamente pela combinação de " + " + ".join(motivos) + "."

    return texto_shap

✅ Classificador XGBoost Treinado.
Recall: 1.0000 | F1-Score: 1.0000


### 📊 Discussão Analítica: Escolha e Impacto das Métricas (Recall e F1-Score)

A avaliação de modelos preditivos no contexto de saúde e triagem médica, como o Guardiã AI, exige um cuidado estrito na escolha das métricas de performance. Por isso, a extração de métricas críticas neste projeto focou primariamente no **Recall** e no **F1-Score**:

**1. A Importância do Recall (Revocação):**
* O Recall mede a proporção de pacientes em estado de `urgencia_critica` (positivos reais) que foram identificados corretamente pelo classificador.
* Em um cenário médico e de segurança, maximizar o Recall é o objetivo principal, pois foca em minimizar os **Falsos Negativos**.

**2. A Importância do F1-Score:**
* O F1-Score representa a média harmônica entre a Precisão (Precision) e o Recall.
* Casos de saúde geralmente possuem desbalanceamento de classes (pacientes saudáveis ou estáveis são muito mais comuns que casos críticos). O F1-Score ajuda a entender o balanço do modelo, garantindo que a alta taxa de Recall não seja alcançada à custa de classificar *absolutamente todos* os pacientes como críticos (o que derrubaria a Precisão e, consequentemente, o F1-Score).

**3. O Impacto dos Erros (Falsos Positivos vs. Falsos Negativos):**
* **Erro Falso Negativo (Risco Crítico):** O modelo classifica um caso grave (como baixo SPO2 ou alta hesitação vocal combinada à taquicardia) como um paciente fora de perigo.
  * *Impacto:* Risco iminente à vida da paciente. Resulta em atraso no atendimento médico emergencial, infração de segurança e consequências fatais.
* **Erro Falso Positivo (Risco Operacional):** O modelo classifica uma paciente estável como tendo alto risco.
  * *Impacto:* Alocação desnecessária de leitos, mobilização indevida de profissionais de emergência e "fadiga de alarmes" nas equipes de saúde. Embora custoso financeiramente e operacionalmente, é muito menos grave que o cenário de Falso Negativo.

# 5. Motores de Extração de Áudio e Texto (Librosa e Whisper)
Define as funções que interagem com o arquivo de áudio. A primeira função calcula a taxa de hesitação vocal através da identificação de intervalos de silêncio acústico (Librosa). A segunda função utiliza o modelo base do Whisper da OpenAI para transcrever o áudio localmente (sem custo de API). Nota: Inclui fallback com dados simulados caso o arquivo mp3 não seja encontrado.

In [6]:
def analisar_audio(audio_path):
    if not os.path.exists(audio_path):
        return {"taxa_hesitacao_porcento": 45.5, "diagnostico_vocal": "Alta Hesitação (Simulado - Arquivo ausente)"}

    y, sr = librosa.load(audio_path, sr=None)
    intervals = librosa.effects.split(y, top_db=30)
    duracao = librosa.get_duration(y=y, sr=sr)
    tempo_ativo = sum([(e - s) / sr for s, e in intervals])
    hesitacao = ((duracao - tempo_ativo) / duracao) * 100 if duracao > 0 else 0
    return {"taxa_hesitacao_porcento": round(hesitacao, 2), "diagnostico_vocal": "Verificado"}

def transcrever_audio(audio_path):
    if not os.path.exists(audio_path):
        return "Patient reports severe chest pain radiating to the left arm, shortness of breath, and feels very anxious."

    model = whisper.load_model("base")
    return model.transcribe(audio_path, language="en")["text"]

# 6. Motor de RAG (Protocolos Clínicos)
Cria uma pequena base vetorial em memória (FAISS) lendo automaticamente todos os Protocolos Clínicos e Diretrizes Terapêuticas (PCDT) armazenados como markdown no Google Drive do usuário. Compara a similaridade da transcrição do paciente com a base de dados para recuperar o protocolo hospitalar mais adequado e cruza as informações utilizando uma LLM (gpt-4o).

In [7]:
def gerar_triagem_rag(transcricao):
    diretorio_pcdt = '/content/drive/MyDrive/PCDTs/'
    arquivos_md = glob.glob(f"{diretorio_pcdt}*.md")

    textos_base = []
    if arquivos_md:
        for arquivo in arquivos_md:
            try:
                with open(arquivo, 'r', encoding='utf-8') as f:
                    textos_base.append(f.read())
            except Exception as e:
                print(f"Erro ao ler {arquivo}: {e}")
    else:
        print("⚠️ Aviso: Nenhum arquivo .md encontrado em /content/drive/MyDrive/PCDTs/. Usando base de contingência.")
        textos_base = [
            "PCDT de Síndrome Coronariana Aguda (SCA): Pacientes com dor torácica irradiada e taquicardia (>110 bpm) exigem ECG em até 10 minutos. Risco iminente de infarto.",
            "PCDT de Transtornos de Ansiedade: Dor atípica, respiração ofegante sem queda de saturação de O2. Sinais vitais compensados."
        ]

    docs = [Document(page_content=t) for t in textos_base]
    vectorstore = FAISS.from_documents(docs, OpenAIEmbeddings(model="text-embedding-3-small"))
    retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

    template = """\
    Baseado na transcrição: {transcricao}
    E no protocolo clínico (PCDT): {contexto}
    Gere um resumo dos dados vitais inferidos. Identifique claramente o nome do protocolo acionado e liste as diretrizes principais extraídas do texto.
    """
    chain = (
        {"contexto": retriever | (lambda docs: "\n".join(d.page_content for d in docs)), "transcricao": RunnablePassthrough()}
        | PromptTemplate.from_template(template)
        | ChatOpenAI(model="gpt-4o", temperature=0.2)
        | StrOutputParser()
    )
    return chain.invoke(transcricao)

# 7. Integração LLM para Prontuário SOAP
Define o template do LangChain responsável por estruturar o Prontuário Médico. O prompt exige que a LLM (GPT-4o) combine três fontes de dados distintas: a transcrição do áudio, as diretrizes clínicas do RAG (PCDT) e a justificativa preditiva (SHAP) do modelo XGBoost, inserindo essas inteligências nas seções corretas do documento SOAP com referência ao protocolo.

In [8]:
def gerar_soap(transcricao, triagem_rag, texto_shap):
    template_soap = """\
    Atue como Auditor IA Médico. Estruture o prontuário SOAP estritamente em Português.

    [Dados de Entrada]
    Transcrição: {transcricao}
    Contexto RAG (Diretrizes do PCDT): {triagem_rag}
    Alerta do ML: {texto_shap}

    [Estrutura Exigida]
    S (Subjetivo): Queixas relatadas.
    O (Objetivo): Dados vitais e físicos.
    A (Avaliação): Hipótese diagnóstica. OBRIGATÓRIO incluir a explicação matemática do alerta: "{texto_shap}". OBRIGATÓRIO referenciar explicitamente o nome do protocolo (PCDT) utilizado.
    P (Plano): Conduta imediata. OBRIGATÓRIO incorporar as diretrizes específicas do PCDT acionado no plano de ação para a tomada de decisão.
    """
    chain = PromptTemplate.from_template(template_soap) | ChatOpenAI(model="gpt-4o", temperature=0.2)
    return chain.invoke({"transcricao": transcricao, "triagem_rag": triagem_rag, "texto_shap": texto_shap}).content

# 8. Orquestração e Execução do Fluxo Principal
Célula final que amarra todos os módulos da aplicação. O fluxo recebe o paciente, processa a parte acústica, simula os sinais vitais da triagem, executa o modelo de classificação para gerar o alerta de risco, aciona o RAG para recuperar o protocolo hospitalar e, por fim, invoca a LLM para redigir o documento consolidado.

In [9]:
audio_path = "/content/CAR0001.mp3"

print("1. Processando Áudio e Transcrição...")
insights_audio = analisar_audio(audio_path)
texto_en = transcrever_audio(audio_path)

print("2. Capturando Dados Estruturados da Triagem Atual...")
dados_triagem = {
    'idade': 45, 'pressao_sistolica': 150, 'pressao_diastolica': 95,
    'frequencia_cardiaca': 122, 'frequencia_respiratoria': 24,
    'temperatura_celsius': 36.8, 'spo2_porcento': 95,
    'taxa_hesitacao_porcento': insights_audio['taxa_hesitacao_porcento'],
    'possui_comorbidade': 1
}

print("3. Executando Modelo Preditivo (XGBoost)...")
explicacao_shap = analisar_risco_shap(dados_triagem)

print("4. Acionando RAG para Mapear Protocolos no Google Drive...")
contexto_rag = gerar_triagem_rag(texto_en)

print("5. Gerando Prontuário Final via Inteligência Generativa...")
with get_openai_callback() as cb:
    prontuario_final = gerar_soap(texto_en, contexto_rag, explicacao_shap)

print("\n" + "="*60)
print("📄 PRONTUÁRIO SOAP CONSOLIDADO")
print("="*60)
print(prontuario_final)

1. Processando Áudio e Transcrição...


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 202MiB/s]


2. Capturando Dados Estruturados da Triagem Atual...
3. Executando Modelo Preditivo (XGBoost)...
4. Acionando RAG para Mapear Protocolos no Google Drive...
5. Gerando Prontuário Final via Inteligência Generativa...

📄 PRONTUÁRIO SOAP CONSOLIDADO
**S (Subjetivo):**  
O paciente, um homem de 39 anos, relata dor no peito que começou na noite anterior. A dor é descrita como constante, aguda e localizada no lado esquerdo do peito. Ele também menciona dificuldade para respirar, sensação de tontura, leve taquicardia e sudorese devido à dificuldade respiratória. O paciente fuma um maço de cigarros por dia há 10-15 anos, consome álcool regularmente (1-2 bebidas por dia) e ocasionalmente usa cannabis. Ele tem um histórico familiar de ataque cardíaco em seu pai aos 45 anos e problemas de colesterol.

**O (Objetivo):**  
- Frequência cardíaca: 122 bpm  
- Pressão diastólica: 95 mmHg  
- Taxa de hesitação: 33.93%  
- Sem condições médicas prévias conhecidas, sem hospitalizações recentes, sem cirurg